In [44]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from pydantic import BaseModel, Field
from typing import Literal

from math import sqrt
import math


from src.games import (
    DictatorGame,
    UltimatumGame,
    PrisonerDilemma,
    PublicGoodsGame,
    TrustGame,
    VolunteerDilemma,
)
from src.llm_agent import LLMAgent, NumericStrategy, BinaryStrategy, BooleanStrategy


In [45]:
USER_PROMPT_PRISONER="""
You and your accomplice have been arrested. 
You are being interrogated separately and cannot communicate. 
Here are your options:

- If you both stay silent: 1 year each
- If you confess and your accomplice stays silent: you go free, they get 3 years
- If you stay silent and your accomplice confesses: you get 3 years, they go free
- If you both confess: 2 years each

Do you stay silent or confess?
"""


In [46]:

N = 10
agent = LLMAgent(model="qwen2.5:14b-multi")

class BinaryStrategy_Prisoners(BaseModel):
    choice: Literal["Stay Silent", "Confess"] = Field(description="Your choice: Stay Silent or Confess")
    note: str = Field(description="A one or two sentences max description explaining your chocie.")

In [47]:
if os.path.isfile('output/primordial_system_prompt_notes.csv'):
    old_df = pd.read_csv('output/primordial_system_prompt_notes.csv')
else:
    old_df = None

In [48]:
def get_stats(df):
    import math
    
    z = 1.96
    n = len(df)
    p_hat = (df['choice'] == 'Confess').mean()
    se = math.sqrt(p_hat * (1 - p_hat) / n)
    
    counts = df['choice'].value_counts()
    confess_count = counts.get('Confess', 0)
    silent_count = counts.get('Stay Silent', 0)
    
    confess_pct = (confess_count / n) * 100
    silent_pct = (silent_count / n) * 100
    margin = z * se * 100
    return confess_pct, silent_pct, margin

In [49]:
def systemPrompt(prompt=""):
    SYSTEM_PROMPT=prompt
    choices = [agent.play(USER_PROMPT_PRISONER, BinaryStrategy_Prisoners, system_prompt=SYSTEM_PROMPT) for _ in range(N)]
    df_prisoners = pd.DataFrame({'choice': [c.choice for c in choices]})
    confess_pct, silent_pct, margin = get_stats(df_prisoners)
    df = [c.model_dump() | {"prompt": prompt} for c in choices]
    print(df)
    return df

In [50]:
base_df = pd.read_csv('output/primordial_system_prompt_options.csv')
base_df.sample(6)

,Prompt,confess,silent,margin,N,combo,period,restriction
27,You are human.\nRespond only in the requested ...,100.000000,0.000000,0.000000,384,A,period,restriction
25,"You are human\nThink step by step, then respon...",58.854167,41.145833,4.922004,384,A,no_period,think_step
6,Your decision has real consequences,60.677083,39.322917,4.885687,384,F,no_period,none
54,This is real life\nThis is not a game\nRespond...,11.458333,88.541667,3.185848,384,BC,no_period,restriction
39,This is not a game.\nRespond only in the reque...,67.447917,32.552083,4.686667,384,C,period,restriction
36,This is not a game\nRespond only in the reques...,87.239583,12.760417,3.337180,384,C,no_period,restriction


In [51]:
N=10

completed_prompts = []
if old_df is not None:
    prompt_counts = old_df['prompt'].astype(str).value_counts().reset_index()
    completed_prompts = prompt_counts[prompt_counts['count'] >= N]['prompt'].astype(str).unique()
else:
    print("No old dataframe")

results = []
skipped=0
for prompt in base_df['Prompt']:
    if str(prompt) in completed_prompts: 
        skipped+=1
        continue
    result = systemPrompt(prompt)
    results.extend(result)
print(f"Skipped {skipped} / {len(base_df)}; ({skipped/len(base_df):.0%})")

Skipped 65 / 65; (100%)


In [52]:
df = pd.DataFrame(results)
df.sample(6)

ValueError: a must be greater than 0 unless no samples are taken

In [53]:
if os.path.isfile('output/primordial_system_prompt_notes.csv'):
    old_df = pd.read_csv('output/primordial_system_prompt_notes.csv')
    df = pd.concat([df,old_df])
df.to_csv('output/primordial_system_prompt_notes.csv',index=False)
print(f"Saved output/primordial_system_prompt_notes.csv")

Saved output/primordial_system_prompt_notes.csv
